# Milestone 3 — Compare Two LLMs for the RAG Pipeline

This notebook is for **Issue #22** only.

Goal:
- compare your current default LLM against one additional LLM
- keep the **retrieved context** and **prompt** fixed for fair comparison
- run **5 queries**
- save the outputs for later discussion in `results/final_discussion.md`


## Workflow

For each query:
1. retrieve documents once from the current RAG backend
2. build one shared context block
3. build one shared prompt
4. send the exact same prompt to two different models
5. save both outputs side by side


In [1]:
import sys
import os
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

from src.rag_pipeline import HybridRAGPipeline, SYSTEM_PROMPT_V2

/Users/lukeni/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paths and artifact checks

In [2]:
processed_dir = PROJECT_ROOT / "data" / "processed"

CORPUS_PATH = processed_dir / "video_games_corpus_final.parquet"
if not CORPUS_PATH.exists():
    csv_fallback = processed_dir / "video_games_corpus_final.csv"
    if csv_fallback.exists():
        CORPUS_PATH = csv_fallback

BM25_INDEX_PATH = processed_dir / "bm25_final_index.pkl"
BM25_TOKENS_PATH = processed_dir / "bm25_final_tokens.pkl"
FAISS_INDEX_PATH = processed_dir / "faiss_final.index"
SEMANTIC_METADATA_PATH = processed_dir / "semantic_final_metadata.pkl"

for p in [CORPUS_PATH, BM25_INDEX_PATH, BM25_TOKENS_PATH, FAISS_INDEX_PATH, SEMANTIC_METADATA_PATH]:
    print(p.name, "->", p.exists())

video_games_corpus_final.csv -> True
bm25_final_index.pkl -> True
bm25_final_tokens.pkl -> True
faiss_final.index -> True
semantic_final_metadata.pkl -> True


## Choose the two models

`PRIMARY_MODEL` should stay equal to your current default model.

For `COMPARE_MODEL`, choose one additional model to test.  
The default below is only a suggestion. If it is unavailable in your Groq account, change it.


In [8]:
PRIMARY_MODEL = os.getenv("LLM_MODEL", "llama-3.1-8b-instant")

# change this to any currently supported Groq model if needed
COMPARE_MODEL = os.getenv("LLM_COMPARE_MODEL", "llama-3.3-70b-versatile")

print("Primary model:", PRIMARY_MODEL)
print("Comparison model:", COMPARE_MODEL)

Primary model: llama-3.1-8b-instant
Comparison model: llama-3.3-70b-versatile


## Queries

In [9]:
queries = [
    "best racing game with fun tracks",
    "story-rich scary game with dark atmosphere",
    "good football game with realistic teams",
    "family-friendly game for casual players",
    "game with strong story and puzzle elements",
]

queries

['best racing game with fun tracks',
 'story-rich scary game with dark atmosphere',
 'good football game with realistic teams',
 'family-friendly game for casual players',
 'game with strong story and puzzle elements']

## Build one RAG pipeline for retrieval and prompt construction

In [10]:
pipeline = HybridRAGPipeline(
    corpus_path=str(CORPUS_PATH),
    bm25_index_path=str(BM25_INDEX_PATH),
    bm25_tokens_path=str(BM25_TOKENS_PATH),
    faiss_index_path=str(FAISS_INDEX_PATH),
    metadata_path=str(SEMANTIC_METADATA_PATH),
    model_name=PRIMARY_MODEL,
    top_k=5,
    system_prompt=SYSTEM_PROMPT_V2,
)

print("Pipeline initialized.")

/Users/lukeni/miniforge3/envs/dsci575-ml/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Pipeline initialized.


## Helper function for one model call

In [11]:
def call_model(model_name: str, prompt: str) -> str:
    llm = ChatGroq(
        model=model_name,
        api_key=os.getenv("GROQ_API_KEY"),
        temperature=0,
    )
    response = llm.invoke(prompt)
    return response.content

## Dry run on one query

In [12]:
test_query = queries[0]

docs = pipeline.retrieve(test_query)
context = pipeline.build_context(docs)
prompt = pipeline.build_prompt(test_query, context)

primary_answer = call_model(PRIMARY_MODEL, prompt)
time.sleep(2)
compare_answer = call_model(COMPARE_MODEL, prompt)

print("=== PRIMARY MODEL ===")
print(primary_answer)

print("\n=== COMPARE MODEL ===")
print(compare_answer)

=== PRIMARY MODEL ===
Based on the retrieved Amazon review context, I would recommend "LittleBigPlanet Karting - Playstation 3" (ASIN: B0050SX00Y) for a fun racing game with enjoyable tracks. The reviews mention that it's a fun game for kids and adults, with customizable carts and a good variety of difficulties.

=== COMPARE MODEL ===
Based on the reviews, I recommend "LittleBigPlanet Karting - Playstation 3" (ASIN: B0050SX00Y) as a fun racing game. Reviewers mention it's a "fun game for everyone" with a "good variety of different difficulties" and allows customization of carts and characters. However, the context does not specifically mention the quality of the tracks.


## Run the full 5-query comparison

In [13]:
rows = []

for query in queries:
    docs = pipeline.retrieve(query)
    context = pipeline.build_context(docs)
    prompt = pipeline.build_prompt(query, context)

    primary_answer = call_model(PRIMARY_MODEL, prompt)
    time.sleep(2)
    compare_answer = call_model(COMPARE_MODEL, prompt)
    time.sleep(2)

    rows.append({
        "query": query,
        "primary_model": PRIMARY_MODEL,
        "compare_model": COMPARE_MODEL,
        "top_titles": " | ".join(docs["product_title"].astype(str).head(5).tolist()),
        "prompt": prompt,
        "primary_answer": primary_answer,
        "compare_answer": compare_answer,
    })

comparison_df = pd.DataFrame(rows)
comparison_df

,query,primary_model,compare_model,top_titles,prompt,primary_answer,compare_answer
0,best racing game with fun tracks,llama-3.1-8b-instant,llama-3.3-70b-versatile,LittleBigPlanet Karting - Playstation 3 | Clas...,\nYou are a careful product recommendation ass...,"Based on the retrieved Amazon review context, ...","Based on the reviews, I recommend ""LittleBigPl..."
1,story-rich scary game with dark atmosphere,llama-3.1-8b-instant,llama-3.3-70b-versatile,Among the Sleep - PlayStation 4 | The Dark Pic...,\nYou are a careful product recommendation ass...,"Based on the retrieved Amazon review context, ...","Based on the reviews, I recommend ""Among the S..."
2,good football game with realistic teams,llama-3.1-8b-instant,llama-3.3-70b-versatile,Backyard Football 2009 - PlayStation 2 | ESPN ...,\nYou are a careful product recommendation ass...,"Based on the retrieved Amazon review context, ...","Based on the reviews, ESPN NFL 2K5 seems to be..."
3,family-friendly game for casual players,llama-3.1-8b-instant,llama-3.3-70b-versatile,"Hey, That's My Fish | Family Game Pack 2001 | ...",\nYou are a careful product recommendation ass...,"Based on the retrieved Amazon review context, ...","Based on the reviews, I recommend ""Hey, That's..."
4,game with strong story and puzzle elements,llama-3.1-8b-instant,llama-3.3-70b-versatile,The Book of Unwritten Tales | Half-Life - Play...,\nYou are a careful product recommendation ass...,"Based on the retrieved Amazon review context, ...","Based on the reviews, I would recommend ""The B..."


## Save the comparison outputs

In [14]:
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / "llm_comparison_results.csv"
comparison_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: /Users/lukeni/Desktop/ubc/MDS/TERM6/575/DSCI_575_project_li0606_Lukeni/results/llm_comparison_results.csv


## Quick comparison view

Read through the outputs and judge:
- which answer is more grounded in the retrieved context
- which answer is clearer and more helpful
- whether the comparison model is actually worse than the current default


In [15]:
comparison_df[["query", "primary_answer", "compare_answer"]]

,query,primary_answer,compare_answer
0,best racing game with fun tracks,"Based on the retrieved Amazon review context, ...","Based on the reviews, I recommend ""LittleBigPl..."
1,story-rich scary game with dark atmosphere,"Based on the retrieved Amazon review context, ...","Based on the reviews, I recommend ""Among the S..."
2,good football game with realistic teams,"Based on the retrieved Amazon review context, ...","Based on the reviews, ESPN NFL 2K5 seems to be..."
3,family-friendly game for casual players,"Based on the retrieved Amazon review context, ...","Based on the reviews, I recommend ""Hey, That's..."
4,game with strong story and puzzle elements,"Based on the retrieved Amazon review context, ...","Based on the reviews, I would recommend ""The B..."


## Decision template

Fill this in after reviewing the results:

- Current default model:
- Compared model:
- Did the new model perform better, worse, or about the same?
- Final decision:
- Reason:

If the comparison model is worse overall, keep your current production model and do not change your backend code.
